# NB15: CPTAC external validation with OpenSlideFM (within-cohort CV)

Within-cohort 5-fold stratified cross-validation on CPTAC OpenSlideFM
patient embeddings. Within each fold the manuscript pipeline
(StandardScaler -> PCA(384) -> Ridge(alpha=30.0) -> Platt) is fitted
on the training fold and applied to the held-out fold.

**Manuscript reference** (Methods + Table 3):
- Pipeline: StandardScaler -> PCA(k=384) -> Ridge(alpha=30.0) -> Platt
- Backbone: OpenSlideFM ViT-L/14 dual-scale, 1,536-dimensional
- HRD threshold for CPTAC: scarHRD >= 42
- CPTAC-LUAD : AUROC = 0.723
- CPTAC-LUSC : AUROC = 0.527
- CPTAC-HNSCC: AUROC = 0.475
- CPTAC-UCEC : evaluated but excluded from primary AUROC reporting

The Ridge regressor uses the continuous Loeffler HRD score as its
regression target; the Platt scaler maps the Ridge output to the
binary HRD label (scarHRD >= 42).

**Required env**: `WORKSPACE`, `CPTAC_OSFM_DIR` (directory containing
`cptac_<cohort>_osfm_patient_embeddings.parquet` per cohort).
**Inputs**: NB13 `labels/cptac_<cohort>_loeffler.csv`.
**Outputs**: `results/cptac_external_osfm/{per_cohort_metrics.csv,
per_cohort_predictions/<cohort>_preds.csv, report.json}`.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
CPTAC_OSFM_DIR = Path(os.environ["CPTAC_OSFM_DIR"]).resolve()
LABELS_DIR = WORKSPACE / "labels"
RESULTS_DIR = WORKSPACE / "results" / "cptac_external_osfm"
PREDS_DIR = RESULTS_DIR / "per_cohort_predictions"
PREDS_DIR.mkdir(parents=True, exist_ok=True)

# manuscript-locked params
SEED = 42
PCA_N = 384
RIDGE_ALPHA = 30.0
BOOT_N = 2000
N_FOLDS = 5
HRD_THRESHOLD_CPTAC = 42

COHORTS = {
    "LUAD":  {"manuscript_AUROC": 0.723, "primary": True},
    "LUSC":  {"manuscript_AUROC": 0.527, "primary": True},
    "HNSCC": {"manuscript_AUROC": 0.475, "primary": True},
    "UCEC":  {"manuscript_AUROC": None, "primary": False},
}

print(f"CPTAC_OSFM_DIR : {CPTAC_OSFM_DIR}")
print(f"PCA_N={PCA_N}, RIDGE_ALPHA={RIDGE_ALPHA}, folds={N_FOLDS}, BOOT_N={BOOT_N}")


In [ ]:
def boot_ci(y, p, fn, n_boot=BOOT_N, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y[idx], p[idx]
        if yb.sum() == 0 or yb.sum() == n:
            continue
        try:
            vals.append(fn(yb, pb))
        except Exception:
            pass
    if not vals:
        return (float("nan"), float("nan"))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def fit_predict_fold(X_tr, X_te, y_tr_cont, y_tr_bin):
    pca_n = min(PCA_N, X_tr.shape[1] - 1, X_tr.shape[0] - 1)
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("pca", PCA(n_components=pca_n, random_state=SEED)),
        ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=SEED)),
    ])
    pipe.fit(X_tr, y_tr_cont)
    z_tr = pipe.predict(X_tr).reshape(-1, 1)
    platt = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=SEED).fit(z_tr, y_tr_bin)
    z_te = pipe.predict(X_te).reshape(-1, 1)
    return platt.predict_proba(z_te)[:, 1]

def evaluate_cohort(cohort):
    feats_path = CPTAC_OSFM_DIR / f"cptac_{cohort.lower()}_osfm_patient_embeddings.parquet"
    labels_path = LABELS_DIR / f"cptac_{cohort.lower()}_loeffler.csv"
    if not feats_path.exists():
        return {"cohort": cohort, "error": f"missing features: {feats_path}"}
    if not labels_path.exists():
        return {"cohort": cohort, "error": f"missing labels: {labels_path}; run NB13"}

    X = pd.read_parquet(feats_path)
    if "patient" in X.columns:
        X = X.set_index("patient")
    X.index = X.index.astype(str).str.upper().str.strip()

    L = pd.read_csv(labels_path)
    L["patient"] = L["patient"].astype(str).str.upper().str.strip()
    L = L.dropna(subset=["HRD_binary"]).drop_duplicates("patient").set_index("patient")

    common = sorted(set(X.index) & set(L.index))
    X = X.loc[common]
    L = L.loc[common]
    y_bin = L["HRD_binary"].astype(int).values
    y_cont = pd.to_numeric(L["HRD_score"], errors="coerce").values
    # if any HRD_score is missing, fall back to binary as Ridge target
    if np.isnan(y_cont).any():
        y_cont = y_bin.astype(float)

    if y_bin.sum() == 0 or y_bin.sum() == len(y_bin):
        return {"cohort": cohort, "error": f"single-class labels (n_pos={int(y_bin.sum())} of {len(y_bin)})"}

    n_splits = min(N_FOLDS, int(y_bin.sum()), int(len(y_bin) - y_bin.sum()))
    if n_splits < 2:
        return {"cohort": cohort, "error": f"too few events for CV (n_pos={int(y_bin.sum())})"}

    Xv = X.values.astype(np.float32)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y_bin), dtype=float)
    for tr, te in skf.split(Xv, y_bin):
        oof[te] = fit_predict_fold(Xv[tr], Xv[te], y_cont[tr], y_bin[tr])

    auc = float(roc_auc_score(y_bin, oof))
    ap = float(average_precision_score(y_bin, oof))
    br = float(brier_score_loss(y_bin, oof))
    auc_lo, auc_hi = boot_ci(y_bin, oof, roc_auc_score)
    ap_lo, ap_hi = boot_ci(y_bin, oof, average_precision_score)

    pd.DataFrame({"patient": common, "y_true": y_bin, "y_score": oof,
                  "HRD_score": L["HRD_score"].values}).to_csv(
        PREDS_DIR / f"{cohort.lower()}_preds.csv", index=False
    )

    return {
        "cohort": cohort, "n": int(len(y_bin)),
        "n_pos": int(y_bin.sum()), "n_neg": int(len(y_bin) - y_bin.sum()),
        "n_splits": int(n_splits),
        "AUROC": auc, "AUROC_lo": auc_lo, "AUROC_hi": auc_hi,
        "AP": ap, "AP_lo": ap_lo, "AP_hi": ap_hi, "Brier": br,
    }


In [ ]:
rows = []
for cohort, cfg in COHORTS.items():
    print(f"\n--- CPTAC-{cohort} (OSFM, {N_FOLDS}-fold CV with PCA-Ridge-Platt) ---")
    res = evaluate_cohort(cohort)
    res["manuscript_AUROC"] = cfg.get("manuscript_AUROC")
    res["primary"] = cfg.get("primary", False)
    if "error" in res:
        print(f"  {res['error']}")
    else:
        ms = cfg.get("manuscript_AUROC")
        ms_str = f" (ms ref {ms:.3f})" if ms else ""
        print(f"  n={res['n']}  HRD+={res['n_pos']}  HRD-={res['n_neg']}  folds={res['n_splits']}")
        print(f"  AUROC = {res['AUROC']:.3f} ({res['AUROC_lo']:.3f}-{res['AUROC_hi']:.3f}){ms_str}")
        print(f"  AP    = {res['AP']:.3f}  Brier = {res['Brier']:.3f}")
    rows.append(res)

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "per_cohort_metrics.csv", index=False)
print()
print(df.to_string(index=False))


In [ ]:
report = {
    "seed": SEED,
    "params": {"PCA_N": PCA_N, "RIDGE_ALPHA": RIDGE_ALPHA},
    "bootstrap_n": BOOT_N,
    "n_folds": N_FOLDS,
    "results": rows,
    "manuscript_refs": {
        "CPTAC_LUAD_OSFM": 0.723,
        "CPTAC_LUSC_OSFM": 0.527,
        "CPTAC_HNSCC_OSFM": 0.475,
    },
}
(RESULTS_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))
